# Case 1 — Clean-data false-positive baseline

**Reproduces:** Fig 4.1 (score histograms), Fig 4.2 (rolling FPR), Table 4.1

No anomalies are injected here — every predicted 'anomaly' is a false positive. The thesis found Transformer-VAE fits the warmup data so tightly that its rolling threshold starts miscalibrated and produces an early false-positive burst before adapting, while MLP-VAE-Cyclic stays calibrated from the start. On the accuracy column below, lower is better — it is literally the false-positive rate.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the real ERA5 slice shipped with the repo under `data/era5/` (16 years, centered on the same warmup/test split used at full scale, split exactly 50:50 warmup/test — see `DATA_LICENSE.md`) — no download needed. Numbers will still differ from the thesis's full-scale figures (much shorter warmup/test period, noisier), but the *qualitative* effect described above should still show up.

**Setup:** this is a private repo, so before running you need a GitHub token as a Colab secret — key icon in the left sidebar -> new secret named `GITHUB_TOKEN`, value = a token from [github.com/settings/tokens](https://github.com/settings/tokens) (read-only `repo` access is enough), then toggle "Notebook access" on.

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: NVIDIA RTX A5000


In [2]:
import os

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Private repo: add a GitHub token as a Colab secret first --
    # key icon in the left sidebar -> Secrets -> new secret named GITHUB_TOKEN,
    # value = a token from github.com/settings/tokens (read-only "repo" access is enough)
    # -> toggle "Notebook access" on.
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    if not os.path.exists("repo"):
        !git clone $clone_url repo
    %cd repo
elif not os.path.exists("run_regression.py"):
    # Already inside a local checkout (e.g. running from notebooks/cases/) --
    # move to the repo root instead of cloning a redundant nested copy.
    %cd ../..

!pip install -q -r requirements.txt


/home/cohenhada/streaming-vae-anomaly-detection

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Run the suite

`notebooks/cases/case01_clean_baseline_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially. Window-mode runs are batched and fast (well under a minute each on GPU); point-mode runs stream one gradient step per row with no batching, so they run on CPU instead (faster than GPU for this access pattern) and take a few minutes each — `modules/stream/point.yaml` caps them to a 5,000-row subset for this reason.

In [3]:
!python run_regression.py notebooks/cases/case01_clean_baseline_suite.yaml \
    --session runs/regression/case01_clean_baseline

[suite] log     : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression/case01_clean_baseline/suite.log
[suite] suite   : /home/cohenhada/streaming-vae-anomaly-detection/notebooks/cases/case01_clean_baseline_suite.yaml
[suite] session : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression/case01_clean_baseline
[suite] base cfg: /home/cohenhada/streaming-vae-anomaly-detection/notebooks/cases/../../modules/era5_common.yaml
[suite] module  : regression.run_trial
[suite] GPUs    : [0, 1]  (2 slot(s))
[suite] 2 run(s) planned:
  [  1] MLP_Cyclic_Clean  [case01_clean_baseline/case01_clean_baseline_suite]  (module: regression.run_trial)
  [  2] TF_VAE_Clean  [case01_clean_baseline/case01_clean_baseline_suite]  (module: regression.run_trial)
[suite] START  'MLP_Cyclic_Clean'  (GPU=0)
[suite] START  'TF_VAE_Clean'  (GPU=1)
GLOBAL SEED = 42
GLOBAL SEED = 42
Logger: Verbosity=3, logging into: '/home/cohenhada/streaming-vae-anomaly-detection/runs/regression/case01_clean_baseli

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [5]:
%env MPLBACKEND=Agg

env: MPLBACKEND=Agg


In [6]:
!python cross_compare.py runs/regression/case01_clean_baseline

[cross] session : runs/regression/case01_clean_baseline
[cross] output  : runs/regression/case01_clean_baseline/cross_compare
[cross] 2 run(s) found:
  MLP_Cyclic_Clean_MLP_Cyclic                         arch=MLP_Cyclic  anomaly=Clean  cyclic=False  mode=window
  TF_VAE_Clean_TF_VAE                                 arch=TF_VAE  anomaly=Clean  cyclic=False  mode=window

[cross] Loading run data ...
  MLP_Cyclic_Clean_MLP_Cyclic ... ok
  TF_VAE_Clean_TF_VAE ... ok

[cross] Writing outputs ...
  saved performance_table.csv
  saved performance_summary.txt

  ══ Clean anomalies ═══════════════════════════════════════════════════════════════════════════════════════════════
  run_name                                     arch       variant              F1    AUC   Prec    Rec    Acc       TP      FP       TN     FN    #params
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────
  TF_VAE_Clean_TF_VAE                          TF_VAE     TF

In [7]:
import pandas as pd
perf = pd.read_csv("runs/regression/case01_clean_baseline/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

,run_name,arch,anomaly,variant,f1,auc,precision,recall,tp,fp,fn
0,TF_VAE_Clean_TF_VAE,TF_VAE,Clean,TF_VAE,NaN,NaN,NaN,NaN,0,110,0
1,MLP_Cyclic_Clean_MLP_Cyclic,MLP_Cyclic,Clean,MLP_Cyclic,NaN,NaN,NaN,NaN,0,335,0


## Read the false-positive rate directly

`accuracy` in the table above is `1 - FPR` here since every sample is labelled normal (no positives exist, so F1/precision/recall are undefined — `fp` and `tp+fp+tn+fn` are what matters). Lower `fp` = better calibration.

In [8]:
perf['fpr_pct'] = 100 * perf['fp'] / (perf['fp'] + perf['tn'])
perf[['run_name', 'arch', 'fp', 'tn', 'fpr_pct']]

,run_name,arch,fp,tn,fpr_pct
0,TF_VAE_Clean_TF_VAE,TF_VAE,110,2806,3.772291
1,MLP_Cyclic_Clean_MLP_Cyclic,MLP_Cyclic,335,2581,11.488340
